# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each record set, field, or column has a unique `@id` in the schema. We'll enumerate top-level record sets, their fields and columns by ID for reference.

> Note: For this dataset, as per schema, typically the primary table is present under a record set. We'll print all record sets and peek inside their schemas.

In [ ]:
# List all record sets by their '@id' and fields
for rs in metadata.record_sets:
    print(f"RecordSet @id: {rs.id if hasattr(rs, 'id') else rs._data.get('@id', '--')}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else rs._data.get('name', '--')}")
    # List all fields/columns in this record set
    if rs.fields:
        for f in rs.fields:
            print(f"    Field @id: {f.id if hasattr(f, 'id') else f._data.get('@id', '--')}")
            print(f"      Name: {f.name if hasattr(f, 'name') else f._data.get('name', '--')}")
    print('---')

### Example record from one record set
Let's print one record from each record set, using its `@id`. Replace IDs in later steps as needed.

In [ ]:
# Print a single example record from each record set
for rs in metadata.record_sets:
    rs_id = rs.id if hasattr(rs, 'id') else rs._data.get('@id', None)
    print(f"RecordSet @id: {rs_id}")
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        print("Sample record:")
        for k, v in recs[0].items():
            print(f"  {k}: {v}")
    else:
        print("No records found for this record set.")
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

Below, we'll get all record set IDs and extract all of them, storing the results as pandas DataFrames in a dictionary.

In [ ]:
# Compile list of record set @ids
record_set_ids = [rs.id if hasattr(rs, 'id') else rs._data.get('@id', None) for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for RecordSet @id {record_set_id}:")
        print(df.columns.tolist())
        print(df.head(2))
    else:
        print(f"No records extracted for RecordSet @id {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
We'll apply common processing steps: filtering, normalizing, and grouping. First, select a numeric field from the main record set by `@id`, then process that field. Please update the chosen `@id` for your analysis as fits your goals.

Below, we'll use the first record set and its first numeric column for demonstration. Update the variables as required for your analysis. All selection and operations reference fields by `@id` or DataFrame column (`@id`).

In [ ]:
# Pick the first available record set and a numeric field for demonstration
main_record_set_id = None
numeric_field_id = None
for rs in metadata.record_sets:
    main_record_set_id = rs.id if hasattr(rs, 'id') else rs._data.get('@id', None)
    # Try to automatically select a float or int field
    for f in rs.fields:
        # Use Croissant's .data_type or ._data['dataType'] field
        dt = getattr(f, 'data_type', f._data.get('dataType', None))
        if dt in ['Float', 'schema:Float', 'Integer', 'schema:Integer', 'Number', 'schema:Number']:
            numeric_field_id = f.id if hasattr(f, 'id') else f._data.get('@id', None)
            break
    if numeric_field_id:
        break

print(f"Chosen record set: {main_record_set_id}")
print(f"Chosen numeric field: {numeric_field_id}")

df = dataframes[main_record_set_id]

# Remove missing/NaN values in numeric field
df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notna()]
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean()  # Example: filter above mean
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): Total {len(filtered_df)}")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to pick a categorical/group field
group_field = None
for f in rs.fields:
    dt = getattr(f, 'data_type', f._data.get('dataType', None))
    if dt in ['Text', 'schema:Text', 'String', 'schema:String']:
        field_id = f.id if hasattr(f, 'id') else f._data.get('@id', None)
        # Ensure field is not all unique
        vals = df[field_id].dropna().unique()
        if len(vals) < 100:  # arbitrary threshold
            group_field = field_id
            break

if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"Grouped mean {numeric_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we'll plot the distribution of the chosen numeric field and a group-wise bar plot if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Grouped bar plot if grouping possible
if group_field:
    group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
    plt.figure(figsize=(10,4))
    sns.barplot(x=group_means.index.astype(str), y=group_means.values)
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we have explored the FAIR² dataset on mass spectrometry analysis of human mast cell extracellular vesicles using the `mlcroissant` library and referenced all record sets and fields by their Croissant `@id`s.

- We loaded the dataset, identified record sets, and extracted records using their `@id`.
- Performed basic exploratory data analysis, including filtering and normalizing a numeric field, and grouping by a categorical field (using only their Croissant `@id`).
- Visualized distributions and group means for rapid insights.

You can continue by adapting field selections, cleaning/processing steps, or visualization options to your analysis use case.